In [48]:
import torch
from torch import nn 

from collections import OrderedDict

from copy import deepcopy

import numpy as np

# Linear Case

In [7]:
torch.manual_seed(1)

x = torch.randn(10, 128)

original_model = nn.Sequential(OrderedDict([
    ("lin1", nn.Linear(128, 16)),
    ("act", nn.ReLU()),
    ("lin2", nn.Linear(16, 32)),
    ("lin3", nn.Linear(32, 16)),
]))

In [8]:
original_model

Sequential(
  (lin1): Linear(in_features=128, out_features=16, bias=True)
  (act): ReLU()
  (lin2): Linear(in_features=16, out_features=32, bias=True)
  (lin3): Linear(in_features=32, out_features=16, bias=True)
)

In [10]:
def merged_two_linear_layers(layer1, layer2):
    
    assert type(layer1) == type(layer2) == nn.Linear
    
    W_merged = layer2.weight @ layer1.weight
    b_merged = layer2.weight @ layer1.bias + layer2.bias

    print("W_merged.shape", W_merged.shape, " : ", "b_merged.shape", b_merged.shape)
    
    out_features, in_features = W_merged.shape
    new = nn.Linear(in_features=in_features, out_features=out_features)
    
    new.weight = nn.Parameter(W_merged) 
    new.bias = nn.Parameter(b_merged)
    
    return new
    
merged_layer = merged_two_linear_layers(original_model.lin2, original_model.lin3)

revised_model = nn.Sequential(
    original_model.lin1,
    original_model.act,
    merged_layer
)

assert torch.allclose(
    revised_model(x),
    original_model(x)
)

print("Assertion Passed!")

W_merged.shape torch.Size([16, 16])  :  b_merged.shape torch.Size([16])
Assertion Passed!


# Convolution
$$
  W_2 * \bigg[A *_{1\times1} \sigma\bigg( (W_1*x) + b_1\bigg) + c\bigg] + b_2
$$

In [52]:
torch.manual_seed(1)

x_conv = torch.randn(10, 128, 10, 10)

original_conv_model = nn.Sequential(OrderedDict([
    ("conv1", nn.Conv2d(128, 64, kernel_size=3)),
    ("act", nn.ReLU()),
    ("conv2", nn.Conv2d(64, 32, kernel_size=1)),
    ("conv3", nn.Conv2d(32, 16, kernel_size=3)),
]))

original_conv_model(x_conv);

In [60]:
def merged_conv_layers(layer1, layer2):
    
    assert type(layer1) == type(layer2) == nn.Conv2d
    assert original_conv_model.conv2.kernel_size == (1, 1)
    

    new_weight = torch.einsum("ij,aibc->ajbc", layer1.weight.squeeze(2, 3), layer2.weight)
    new_bias =  torch.einsum("i,aibc->a", layer1.bias, layer2.weight) + layer2.bias
    
    # remark: we have to make sure that stride and other paramer
    new_module = deepcopy(layer2)
    

    
    new_module.weight = nn.Parameter(new_weight)
    new_module.bias = nn.Parameter(new_bias)
    
    return new_module

merged_conv = merged_conv_layers(original_conv_model.conv2, original_conv_model.conv3)


revised_conv_model = nn.Sequential(
    original_conv_model.conv1,
    original_model.act,
    merged_conv
)

with torch.no_grad():
    np.testing.assert_allclose(
        revised_conv_model(x_conv),
        original_conv_model(x_conv),
        atol=1e-6
    )

In [56]:
original_conv_model.conv2.weight.squeeze(2, 3).shape

torch.Size([32, 64])